# Computational Modelling Workshop

This computatioanl modelling workshop will introduce the NEURON computational modelling environment in Python.
Participants will gain experience working with a multiple-compartment retinal ganglion cell neuron.

## Learning Objectives
1.  Gain an overview of how the NEURON Computational Modelling environment works
2. Identify the effect of altering sodium and potassium channels on neuronal excitability
3. Observe how different stimulating locations differentially modulates neuronal behaviour

## Setup
Run these two cells first, in order. The first installs NEURON and compiles the ion-channel mechanisms; the second imports NEURON, so it must come after.

If anything misbehaves, use **Runtime → Restart session** and run again from the top.

In [ ]:
#@title ▶ Run this first: installs NEURON and fetches the workshop files (~1 min) { display-mode: "form" }
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"   # <-- change to your repository

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install NEURON and interactive-plot support
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neuron==9.0.2", "ipympl"], check=True)
    from google.colab import output
    output.enable_custom_widget_manager()

    # Fetch the workshop files (.mod, morphology, utils) and move into that folder
    repo_dir = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "-q", REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)

# Compile the .mod files (skipped if already compiled; delete the x86_64/arm64 folder to force a rebuild)
if not any(os.path.isdir(d) for d in ("x86_64", "arm64", "aarch64")):
    build = subprocess.run(["nrnivmodl"], capture_output=True, text=True)
    if build.returncode != 0:
        print(build.stdout[-3000:], build.stderr[-3000:])
        raise RuntimeError("Compiling the .mod files failed (see messages above)")

print("Setup complete. Working directory:", os.getcwd())


In [ ]:
%matplotlib widget
# Interactive figures (zoom / pan / reset buttons). Needs ipympl: pip install ipympl
from neuron import h, gui
import numpy as np
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

PEAK_HEIGHT = -20.0      # mV : spike-detection threshold
TIME_RESOLUTION = 0.05   # ms

plt.rcParams['figure.dpi'] = 100   # widget figures are sized in screen pixels
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Overview of the NEURON Computational Modelling Environment
### Single-compartment model
A single compartment with the same geometry and ion channels as the soma of the multi-compartment cell (with a larger leak to stand in for the missing dendrites and axon), stimulated with an intracellular current clamp (IClamp).

In [ ]:
# A single cylindrical compartment with the soma parameters of ctt3219b.hoc
def build_soma_sc():
    soma_sc = h.Section(name='soma_sc')
    soma_sc.L    = 27.69     # µm (gives the same membrane area as the soma, ~1216 µm²)
    soma_sc.diam = 13.98     # µm
    soma_sc.nseg = 1
    soma_sc.Ra   = 110       # ohm·cm
    soma_sc.cm   = 1         # µF/cm²

    # Passive leak
    # The full model uses g_pas = 8e-6 S/cm², but on an isolated soma that gives an input
    # resistance of ~10 GΩ (−10 pA -> ~−100 mV). In the full cell, the dendrites and axon
    # carry most of that current away. A larger leak stands in for them, giving an
    # input resistance of ~200 MΩ, which is physiological for an RGC.
    soma_sc.insert('pas')
    soma_sc.g_pas = 0.0004    # S/cm²
    soma_sc.e_pas = -62.5     # mV

    # Active channels (spike.mod) and calcium pump (capump.mod)
    soma_sc.insert('spike')
    soma_sc.insert('cad')
    soma_sc.gnabar_spike = 0.080     # S/cm² : Na
    soma_sc.gkbar_spike  = 0.018     # S/cm² : delayed-rectifier K
    soma_sc.gabar_spike  = 0.054     # S/cm² : A-type K
    soma_sc.gcabar_spike = 0.0015    # S/cm² : Ca
    soma_sc.gkcbar_spike = 0.000065  # S/cm² : Ca-activated K
    soma_sc.ena = 35.0               # mV
    soma_sc.ek  = -75.0              # mV
    soma_sc(0.5).depth_cad = soma_sc.diam / 2   # µm : Ca shell depth
    return soma_sc

soma_sc = build_soma_sc()


### Apply a current clamp and run
`CCINTENSITY` is a list of step amplitudes (pA); the cell runs one simulation and draws one figure per amplitude. `CCDURATION` (ms) sets the length of the current step, which starts 100 ms into a 500 ms simulation.

In [ ]:
# Current-clamp parameters
CCINTENSITY = [100, 200, 400]   # pA : one simulation (and one figure) per intensity
CCDURATION  = 1     # ms
cc_delay    = 100   # ms

h.dt      = TIME_RESOLUTION
h.tstop   = 500     # ms
h.celsius = 33

# Rebuild the compartment if Section 3 has removed it
if 'soma_sc' not in globals():
    soma_sc = build_soma_sc()

sc_iclamp = h.IClamp(soma_sc(0.5))
sc_iclamp.delay = cc_delay
sc_iclamp.dur   = CCDURATION

# Recording vectors
sc_t = h.Vector().record(h._ref_t)
sc_v = h.Vector().record(soma_sc(0.5)._ref_v)
sc_m = h.Vector().record(soma_sc(0.5)._ref_m_spike)
sc_h = h.Vector().record(soma_sc(0.5)._ref_h_spike)
sc_n = h.Vector().record(soma_sc(0.5)._ref_n_spike)
sc_i = h.Vector().record(sc_iclamp._ref_i)

plt.close('all')   # close figures from previous runs

for intensity in CCINTENSITY:
    sc_iclamp.amp = intensity / 1000   # IClamp takes nA

    # Run
    h.finitialize(-62.5)
    soma_sc.cai = 0.0001   # reset calcium to its resting concentration
    h.fcurrent()
    h.frecord_init()
    h.continuerun(h.tstop)

    t_sc = np.array(sc_t)
    v_sc = np.array(sc_v)
    peaks, _ = find_peaks(v_sc, height=PEAK_HEIGHT)
    print(f"{intensity} pA -> Spikes: {len(peaks)}  ({len(peaks) / (CCDURATION / 1000):.1f} Hz during the pulse)")

    # Plot
    fig, axes = plt.subplots(3, 1, figsize=(8, 3), sharex=True,
                             gridspec_kw={'height_ratios': [1, 1, 1]})
    fig.canvas.toolbar_position = 'right'   # keep the zoom/pan toolbar off the axis labels
    fig.canvas.header_visible = False
    fig.suptitle(f'IClamp = {intensity} pA')

    axes[0].plot(t_sc, np.array(sc_i) * 1000, color='k')
    axes[0].set_ylabel('IClamp (pA)')

    axes[1].plot(t_sc, v_sc, lw=1)
    axes[1].set_ylabel('Vm (mV)')

    axes[2].plot(t_sc, np.array(sc_m), label='m (Na act.)')
    axes[2].plot(t_sc, np.array(sc_h), label='h (Na inact.)')
    axes[2].plot(t_sc, np.array(sc_n), label='n (K act.)')
    axes[2].set_ylabel('Gate')
    axes[2].set_xlabel('Time (ms)')
    axes[2].legend(loc='upper right')

    plt.tight_layout()
    plt.show()

## 2. Altering sodium and potassium channels
### Sodium conductance sweep
A 200 pA current step is applied for 300 ms while the maximal Na conductance (`gnabar_spike`) is scaled by the multiplier `NA_X`, from 0 (no Na channels) to 2 (double the default).

In [ ]:
# Current-clamp parameters
CCINTENSITY = 300   # pA
CCDURATION  = 300   # ms
cc_delay    = 100   # ms

# Na conductance multipliers
NA_X = np.arange(0, 11, 1)   # 0, 0.25, ..., 2

h.dt      = TIME_RESOLUTION
h.tstop   = 500     # ms
h.celsius = 33

# Start from a freshly built compartment so the default conductances are used
if 'soma_sc' in globals():
    h.delete_section(sec=soma_sc)
soma_sc = build_soma_sc()
GNA_DEFAULT = soma_sc.gnabar_spike   # S/cm²

sc_iclamp = h.IClamp(soma_sc(0.5))
sc_iclamp.delay = cc_delay
sc_iclamp.dur   = CCDURATION
sc_iclamp.amp   = CCINTENSITY / 1000   # IClamp takes nA

# Recording vectors
sc_t = h.Vector().record(h._ref_t)
sc_v = h.Vector().record(soma_sc(0.5)._ref_v)

na_traces = {}   # NA_X -> Vm trace
na_spikes = {}   # NA_X -> number of spikes

for na_x in NA_X:
    soma_sc.gnabar_spike = GNA_DEFAULT * na_x

    h.finitialize(-62.5)
    soma_sc.cai = 0.0001   # reset calcium to its resting concentration
    h.fcurrent()
    h.frecord_init()
    h.continuerun(h.tstop)

    v = np.array(sc_v)
    peaks, _ = find_peaks(v, height=PEAK_HEIGHT, prominence=10)   # prominence ignores ripples on a plateau
    na_traces[na_x] = v
    na_spikes[na_x] = len(peaks)
    print(f"NA_X = {na_x:.2f} (gNa = {GNA_DEFAULT * na_x:.3f} S/cm²) -> Spikes: {len(peaks)}  "
          f"({len(peaks) / (CCDURATION / 1000):.1f} Hz)")

t_na = np.array(sc_t)
soma_sc.gnabar_spike = GNA_DEFAULT   # restore the default for other sections

In [ ]:
plt.close('all')   # close figures from previous runs

# Vm for every multiplier (axes are linked: zooming one zooms all)
fig, axes = plt.subplots(len(NA_X), 1, figsize=(8, 1.1 * len(NA_X)), sharex=True, sharey=True)
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
for ax, na_x in zip(axes, NA_X):
    ax.plot(t_na, na_traces[na_x], lw=0.8)
    ax.set_ylabel(f'×{na_x:.2f}', rotation=0, ha='right', va='center')
axes[0].set_title('Vm (mV) for each Na conductance multiplier')
axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

# Spike count vs multiplier
fig2, ax2 = plt.subplots(figsize=(5, 3))
fig2.canvas.toolbar_position = 'right'
fig2.canvas.header_visible = False
ax2.plot(NA_X, [na_spikes[x] for x in NA_X], 'o-')
ax2.set_xlabel('NA_X (multiplier on gNa)')
ax2.set_ylabel(f'Spikes in {CCDURATION} ms')
plt.tight_layout()
plt.show()

## 3. Multi-compartment model with extracellular stimulation
### Set up the model
Load the morphology, insert the extracellular mechanisms and position the stimulating electrode.

In [ ]:
"""from sheasbyfohlmeister1999_utils import update_rx_3d, plot_final_rgc_with_electrode

PLOT_ELECTRODE_POSITION = False

# Remove the single-compartment model from Section 1 so it is not part of this cell
if 'soma_sc' in globals():
    h.delete_section(sec=soma_sc)
    del soma_sc, sc_iclamp

# Load the model
h.load_file("sheasbyfohlmeister1999/MediumComplex/CTT3219B/ctt3219b.asc")
h.load_file("sheasbyfohlmeister1999/MediumComplex/CTT3219B/ctt3219b.hoc")

# Reverse Z coordinates to flip the morphology (soma facing up)
for sec in h.allsec():
    n3d = int(h.n3d(sec=sec))
    if n3d < 2: continue
    xs = [h.x3d(i, sec=sec) for i in range(n3d)]
    ys = [h.y3d(i, sec=sec) for i in range(n3d)]
    zs = [h.z3d(i, sec=sec) for i in range(n3d)]
    ds = [h.diam3d(i, sec=sec) for i in range(n3d)]
    sec.pt3dclear()
    for i in range(n3d):
        sec.pt3dadd(xs[i], ys[i], -zs[i], ds[i])
h.define_shape()

# Centre the cell on the soma
n_soma = int(h.n3d(sec=h.soma))
avg_x = np.mean([h.x3d(i, sec=h.soma) for i in range(n_soma)])
avg_y = np.mean([h.y3d(i, sec=h.soma) for i in range(n_soma)])
avg_z = np.mean([h.z3d(i, sec=h.soma) for i in range(n_soma)])

for sec in h.allsec():
    # Insert extracellular and xtra for field stimulation
    sec.insert('extracellular')
    sec.insert('xtra')
    sec.insert('pas')

    n = int(h.n3d(sec=sec))
    if n < 2:
        continue
    xs = [h.x3d(i, sec=sec) for i in range(n)]
    ys = [h.y3d(i, sec=sec) for i in range(n)]
    zs = [h.z3d(i, sec=sec) for i in range(n)]
    ds = [h.diam3d(i, sec=sec) for i in range(n)]
    sec.pt3dclear()
    for i in range(n):
        sec.pt3dadd(xs[i] - avg_x, ys[i] - avg_y, zs[i] - avg_z, ds[i])

    # Map the NEURON extracellular mechanisms to the xtra pointers
    for seg in sec:
        h.setpointer(seg._ref_e_extracellular, 'ex', seg.xtra)
        h.setpointer(seg._ref_i_membrane, 'im', seg.xtra)

h.ko0_k_ion = 5.0    # mM (extracellular)
h.ki0_k_ion = 145.0  # mM (intracellular)
h.define_shape()

# Electrode position (µm)
el_x, el_y, el_z = 0, 0, 25 + h.soma.diam/2 + 1   # +1 to account for ILM thickness
#el_x, el_y, el_z = 200, 0, 25 + h.soma.diam/2 + 1 # Distal Axon Stimulation
update_rx_3d(el_x, el_y, el_z)

if PLOT_ELECTRODE_POSITION:
    plot_final_rgc_with_electrode(el_x, el_y, el_z)"""

### Define the stimulus
A single 10 ms cathodic (negative) current pulse is delivered through the extracellular electrode, starting 100 ms into a 500 ms simulation.

In [ ]:
"""# Stimulus parameters
stim_amplitude = 10     # uA (cathodic, applied as a negative current)
stim_delay     = 100    # ms
stim_duration  = 10     # ms

h.dt     = TIME_RESOLUTION
h.tstop  = 500          # ms
h.celsius = 33

# Build the cathodic square pulse and play it into the extracellular electrode
t_stim = np.arange(0, h.tstop + h.dt, h.dt)
stim_values = np.zeros_like(t_stim)
stim_values[(t_stim >= stim_delay) & (t_stim < stim_delay + stim_duration)] = -stim_amplitude

stim_vec   = h.Vector(stim_values)
t_vec_stim = h.Vector(t_stim)
stim_vec.play(h._ref_is_xtra, t_vec_stim)

# Recording vectors
t_vec = h.Vector().record(h._ref_t)
v_rec = {name: h.Vector().record(sec(0.5)._ref_v)
         for name, sec in [('soma', h.soma), ('initseg', h.initseg), ('axon', h.axon)]}
m_soma = h.Vector().record(h.soma(0.5)._ref_m_spike)
h_soma = h.Vector().record(h.soma(0.5)._ref_h_spike)
n_soma = h.Vector().record(h.soma(0.5)._ref_n_spike)"""

### Run the simulation

In [ ]:
"""h.finitialize(-62.5)

# Reset calcium to its resting concentration (or else repeated runs will differ)
for sec in h.allsec():
    if h.ismembrane('ca_ion', sec=sec):
        sec.cai = 0.0001
h.fcurrent()
h.frecord_init()

h.continuerun(h.tstop)

t = np.array(t_vec)
v = {name: np.array(vec) for name, vec in v_rec.items()}

peaks, _ = find_peaks(v['soma'], height=PEAK_HEIGHT)
print(f"Somatic spikes: {len(peaks)}  at t = {np.round(t[peaks], 2)} ms")"""

### Plot the results

In [ ]:
"""plt.close('all')   # close figures from previous runs
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True,
                         gridspec_kw={'height_ratios': [1, 3, 2]})
fig.canvas.toolbar_position = 'right'   # keep the zoom/pan toolbar off the axis labels
fig.canvas.header_visible = False

axes[0].plot(t_stim, stim_values, color='k')
axes[0].set_ylabel('Stimulus (µA)')

for name, trace in v.items():
    axes[1].plot(t, trace, label=name, lw=1)
axes[1].set_ylabel('Vm (mV)')
axes[1].legend(loc='upper right')

axes[2].plot(t, np.array(m_soma), label='m (Na act.)')
axes[2].plot(t, np.array(h_soma), label='h (Na inact.)')
axes[2].plot(t, np.array(n_soma), label='n (K act.)')
axes[2].set_ylabel('Gate (soma)')
axes[2].set_xlabel('Time (ms)')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()"""